# build_fewshot_exemplars
Reads the frozen train/test splits (does NOT create splits). Assigns per test row: 5 domain exemplars (train instructions, ref-stripped, FULL — no cap) and 5 mismatched exemplars (non-device rows from grit-entity-new.jsonl). Seeded; K=2 uses the first 2. Writes fewshot_exemplars.jsonl.

In [ ]:
from google.colab import drive
import os, json, re, random
from collections import defaultdict
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"

# Drops every sentence containing an <img> pointer. 13 of the 2,111 unique domain exemplars
# (16 of 2,440 instances, 0.6%) end up with no literal device keyword because the keyword
# occurred only in the dropped pointer sentences. Checked against the source train rows:
# 13 of 13 still describe the device in the surviving text under an inflected form the
# keyword list does not match ("computer mice" for "computer mouse"), so the exemplars stay
# on-domain. The mismatched pool is matched on the raw instruction and is unaffected: 0 of
# 2,440 mismatched exemplars contain a device keyword.
def strip_refs(instr):
    return " ".join(p for p in re.split(r"(?<=\.)\s+", instr) if "<img>" not in p).strip()

# domain pool = frozen train instructions (ref-stripped, FULL)
train = [json.loads(l) for l in open(f"{DRIVE}/train.jsonl", encoding="utf-8")]
dom_by_t = defaultdict(list)
for r in train: dom_by_t[r["output_image"]].append(strip_refs(r["instruction"]))
dom_targets = list(dom_by_t.keys())

# mismatched pool = non-device rows from grit-entity-new.jsonl (same source).
# use the Drive copy if present; otherwise download it (cached) exactly as data.ipynb did.
KW = ["desktop computer","computer setup","laptop","tablet","smartphone","mobile phone","cell phone",
      "headphones","earphones","earbuds","keyboard","computer mouse","game controller","gaming controller",
      "gaming console","vr headset","virtual reality headset"]
pat = re.compile("|".join(re.escape(k) for k in KW), re.I)
grit = f"{DRIVE}/grit-entity-new.jsonl"
if not os.path.exists(grit):
    from huggingface_hub import hf_hub_download
    import shutil
    dl = hf_hub_download(repo_id="yzwang/X2I-subject-driven", filename="grit-entity-new.jsonl", repo_type="dataset")
    shutil.copy(dl, grit)   # saved to Drive so future runs skip the download
print("grit source:", grit)
mis_by_t = defaultdict(list)
for line in open(grit, encoding="utf-8"):
    r = json.loads(line); d = r["output_image"].split("/")[0]
    if d in ("character", "human"): continue
    if not pat.search(r["instruction"]): mis_by_t[r["output_image"]].append(strip_refs(r["instruction"]))
    if sum(len(v) for v in mis_by_t.values()) >= 20000: break
mis_targets = list(mis_by_t.keys())
print(f"pools: domain {len(dom_targets)} targets | mismatched {len(mis_targets)} targets")

test = [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]
random.seed(42)
# write-once: the five few-shot arms are keyed to these exemplars. Re-running reproduces the
# file byte-for-byte from the frozen splits, but a silent overwrite is not worth the risk.
if os.path.exists(f"{DRIVE}/fewshot_exemplars.jsonl"):
    raise SystemExit(f"exists, kept: {DRIVE}/fewshot_exemplars.jsonl")
with open(f"{DRIVE}/fewshot_exemplars.jsonl", "w", encoding="utf-8") as f:
    for r in test:
        uid = r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
        dom = [random.choice(dom_by_t[t]) for t in random.sample(dom_targets, 5)]   # 5 DISTINCT-target scenes
        mis = [random.choice(mis_by_t[t]) for t in random.sample(mis_targets, 5)]
        f.write(json.dumps({"uid": uid, "domain": dom, "mismatched": mis}) + "\n")
print(f"wrote {len(test)} rows -> {DRIVE}/fewshot_exemplars.jsonl (splits untouched)")

Mounted at /content/drive
grit source: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/grit-entity-new.jsonl
pools: domain 8059 targets | mismatched 5583 targets
wrote 488 rows -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/fewshot_exemplars.jsonl (splits untouched)
